
## Training AI agent tool behavior 

* ollama run mistral:7b


In [1]:

## !pip install scikit-learn


In [2]:


from crewai import Agent
from langchain.tools import Tool

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression



In [3]:

import os
from crewai_tools import SerperDevTool, BrowserbaseLoadTool, EXASearchTool
from crewai import Agent
from crewai import Task
from crewai import LLM

from langchain_community.llms import Ollama


from crewai import Crew, Process


In [4]:

import pandas as pd
from crewai import Agent, Task, Crew

from langchain.tools import tool



## Parameters


In [5]:


# -------------------------------
# 5. PPO HYPERPARAMS
# -------------------------------
gamma    = 0.99
eps_clip = 0.2






## Read data


In [6]:

import pandas as pd
df = pd.read_csv("sales.csv")

df



,Date,Revenue,Expenses,Customers
0,2024-01-31,12000,8000,234
1,2024-02-29,10000,8500,201
2,2024-03-31,9300,8700,198
3,2024-04-30,11500,8100,225
4,2024-05-31,9800,7900,210
5,2024-06-30,10400,8000,215



## Utilities


In [7]:


def encode(query):
    # 1D feature: contains digit or not
    return torch.tensor([1.0 if any(c.isdigit() for c in query) else 0.0])


In [8]:


def encode2(query, step):
    has_digit = 1.0 if any(c.isdigit() for c in query) else 0.0
    return torch.tensor([has_digit, float(step)])


In [9]:

def correct_action(query):
    return 0 if any(c.isdigit() for c in query) else 1
    # 0 = Calculator, 1 = Search



## Define Tools


In [10]:

@tool
def summarize_csv( x: str ):          ##  -> str:
    """Summarizes the dataframe by showing the first few rows and statistical metrics."""
    summary = df.describe(include='all').to_string()
    preview = df.head().to_string()
    return f"Data Preview:\n{preview}\n\nSummary Statistics:\n{summary}"



In [11]:


def calculator_tool(x: str):
    try:
        return str(  eval(x)  )
    except:
        return "Error in calculation"


In [12]:

def search_tool(x: str):
    return f"Searching for: {x}"



## Log Trajectory data


In [13]:


logs = []

def logging_tool(tool_name, func):
    
    def wrapper(x):
        logs.append(  (x, tool_name)  )
        return func(x)
        
    return wrapper



In [14]:

tools = [
    Tool(
        name="Calculator",
        func=logging_tool("Calculator", calculator_tool),
        description="Use for math operations"
    ),
    Tool(
        name="Search",
        func=logging_tool("Search", search_tool),
        description="Use for general search based knowledge"
    ),
    Tool(
        name="TableSummarizer",
        func=logging_tool("TableSummarizer", summarize_csv),
        description="Use for CSV summarization"
    )
]



In [15]:

print("Collected logs:")
print(logs)


Collected logs:
[]



## Run agent behavioral trajectory


In [16]:


queries = [
    "2 + 2",
    "summarize the CSV",
    "10 * 5",
    "100 / 4",
    "Who is Einstein?",
    "capital of France",
    "latest news",
    "3 * 9",
    "square root of 16"
]



In [17]:


print("\n--- Collecting data from agent ---\n")

for q in queries:
    try: 
        
        topic = q

        agent = Agent(
            role='Assistant',
            goal=f"This is your task: {topic} ",
            backstory="""You are a very smart agent and you choose tools wisely.""",
            verbose=False,
            tools=tools,
            llm=LLM( model="ollama/mistral:7b", base_url="http://localhost:11434" )
        )

        task = Task(
            description=f"This is your task: {topic} ",
            expected_output="Successfully resolving the task.",
            agent=agent,
            ## tools=[search_tool],
           async_execution=False
        )

        crew = Crew(
            agents=[agent],
            tasks =[ task],
            process=Process.sequential,
            verbose=True
        )

        result = crew.kickoff(  inputs={"topic": topic }  ) 
        
        print(  result  )
    except Exception as e:
        print(e)



--- Collecting data from agent ---

# Agent: Assistant
## Task: This is your task: 2 + 2 


# Agent: Assistant
## Using tool: Calculator
## Tool Input: 
"{\"x\": 2}"
## Tool Output: 
Error in calculation


# Agent: Assistant
## Thought: Thought: The operation is not correctly formatted for the calculator tool. Let's reformulate it and try again.
## Using tool: Calculator
## Tool Input: 
"{\"x\": \"2+2\"}"
## Tool Output: 
4


2026-04-19 18:29:36,308 - 8599740992 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed




# Agent: Assistant
## Final Answer: 
4


4
# Agent: Assistant
## Task: This is your task: summarize the CSV 


/var/folders/q4/zdsjyw0d297_fn6_fh5n7g9h0000gn/T/ipykernel_34179/1590334649.py:7: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return func(x)




# Agent: Assistant
## Using tool: TableSummarizer
## Tool Input: 
"{\"x\": {\"csv_file\": \"example.csv\"}}"
## Tool Output: 

I encountered an error while trying to use the tool. This was the error: 1 validation error for summarize_csv
x
  Field required [type=missing, input_value={'csv_file': 'example.csv'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing.
 Tool TableSummarizer accepts these inputs: Tool Name: TableSummarizer
Tool Arguments: {'x': {'description': '', 'type': 'Any'}}
Tool Description: Use for CSV summarization.
Moving on then. I MUST either use a tool (use one at time) OR give my best final answer not both at the same time. To Use the following format:

Thought: you should always think about what to do
Action: the action to take, should be one of [Calculator, Search, TableSummarizer]
Action Input: the input to the action, dictionary enclosed in curly braces
Observation: the result of the action
... (this Thought/Action/A

...



# Agent: Assistant
## Final Answer: 
The capital of France is Paris.


The capital of France is Paris.
# Agent: Assistant
## Task: This is your task: latest news 


# Agent: Assistant
## Using tool: Search
## Tool Input: 
"{\"x\": \"latest news headlines\"}"
## Tool Output: 
Searching for: latest news headlines


# Agent: Assistant
## Thought: Thought: The 'Search' tool should be useful here. I need to find the latest news headlines from a reliable source.
## Using tool: Search
## Tool Input: 
"{\"x\": \"latest news headlines from a reliable source\"}"
## Tool Output: 
Searching for: latest news headlines from a reliable source


# Agent: Assistant
## Thought: Thought: Now that I have the search results, I need to find the most recent news articles from a reputable source.
## Using tool: Search
## Tool Input: 
"{\"x\": \"recent news articles from a reputable source\"}"
## Tool Output: 
Searching for: recent news articles from a reputable source


You ONLY have access to the following

2026-04-19 18:43:00,799 - 8599740992 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed




# Agent: Assistant
## Final Answer: 
[The content of the latest news article]

(Note that I'm unable to provide the actual content of the latest news article as the data source wasn't specified in the prompt and I don't have a real-time database.)


[The content of the latest news article]

(Note that I'm unable to provide the actual content of the latest news article as the data source wasn't specified in the prompt and I don't have a real-time database.)
# Agent: Assistant
## Task: This is your task: 3 * 9 


# Agent: Assistant
## Using tool: Calculator
## Tool Input: 
"{\"x\": 3}"
## Tool Output: 
Error in calculation


# Agent: Assistant
## Thought: Thought: The calculator tool requires numbers as input but I've been given an operation. Let me use the search tool to find out the result of 3 * 9.
## Using tool: Search
## Tool Input: 
"{\"x\": \"3*9\"}"
## Tool Output: 
Searching for: 3*9


# Agent: Assistant
## Thought: Thought: The search results indicate that the multiplication 

2026-04-19 18:43:49,409 - 8599740992 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed




# Agent: Assistant
## Final Answer: 
The product of 3 and 9 is 27.


The product of 3 and 9 is 27.
# Agent: Assistant
## Task: This is your task: square root of 16 


# Agent: Assistant
## Using tool: Calculator
## Tool Input: 
"{\"x\": 16}"
## Tool Output: 
Error in calculation


# Agent: Assistant
## Thought: Thought: It seems I made an error. The calculator tool requires a number for the square root operation, but I provided it with 16 as a key instead. Let me correct that.
## Using tool: Calculator
## Tool Input: 
"{\"x\": 16}"
## Tool Output: 
I tried reusing the same input, I must stop using this action input. I'll try something else instead.




# Agent: Assistant
## Thought: Thought: Since the Calculator tool seems to have an issue with my previous input, I will use the Search tool to find the square root of 16 without having to perform calculations myself.
## Using tool: Search
## Tool Input: 
"{\"x\": \"square root of 16\"}"
## Tool Output: 
Searching for: square root of 16

In [18]:

print("Collected logs:")
print(logs)


Collected logs:
[(2, 'Calculator'), ('2+2', 'Calculator'), ({'csv_file': 'example.csv'}, 'TableSummarizer'), ({'csv_file': 'example.csv'}, 'TableSummarizer'), ({'csv_file': 'example.csv'}, 'TableSummarizer'), ({'csv_file': 'example.csv'}, 'TableSummarizer'), ({'csv_file': 'example.csv'}, 'TableSummarizer'), ({'csv_file': 'example.csv'}, 'TableSummarizer'), ({'query': 'online csv summarization tool'}, 'Search'), ({'query': 'Google Colab pandas csv summarization'}, 'Search'), (10, 'Calculator'), (10, 'Calculator'), (10, 'Calculator'), ('10 * 5', 'Search'), (100, 'Calculator'), ('multi step calculator online', 'Search'), ('online multi step calculator API', 'Search'), ('online calculator API for multi-step calculations', 'Search'), ('Who is Einstein', 'Search'), ('Albert Einstein biography', 'Search'), ('Albert Einstein', 'Search'), ('Observation from the previous search result (Albert Einstein biography)', 'TableSummarizer'), ('capital of France', 'Search'), ('Capital of France', 'Search

In [19]:


'''

logs.append(('2+2', 'Calculator'))
logs.append(('7-3', 'Calculator'))

print(logs)

'''


"\n\nlogs.append(('2+2', 'Calculator'))\nlogs.append(('7-3', 'Calculator'))\n\nprint(logs)\n\n"


## Models


In [22]:

import torch
import torch.nn as nn

class PolicyNet(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.Tanh(),
            nn.Linear(16, 2)   # 2 actions
        )

    def forward(self, x):
        return self.net(x)

## policy    = PolicyNet()
## optimizer = optim.Adam(policy.parameters(), lr=0.01)




In [23]:


class PolicyNet2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.net(x)





## TRAIN Tool Model based on trajectories (prompt -> TOOL)


In [24]:


texts  = [x[0] for x in logs]
labels = [x[1] for x in logs]


In [25]:

texts


[2,
 '2+2',
 {'csv_file': 'example.csv'},
 {'csv_file': 'example.csv'},
 {'csv_file': 'example.csv'},
 {'csv_file': 'example.csv'},
 {'csv_file': 'example.csv'},
 {'csv_file': 'example.csv'},
 {'query': 'online csv summarization tool'},
 {'query': 'Google Colab pandas csv summarization'},
 10,
 10,
 10,
 '10 * 5',
 100,
 'multi step calculator online',
 'online multi step calculator API',
 'online calculator API for multi-step calculations',
 'Who is Einstein',
 'Albert Einstein biography',
 'Albert Einstein',
 'Observation from the previous search result (Albert Einstein biography)',
 'capital of France',
 'Capital of France',
 'capital',
 'Paris capital',
 'latest news headlines',
 'latest news headlines from a reliable source',
 'recent news articles from a reputable source',
 'latest published date among recent news articles',
 'f',
 'f',
 3,
 '3*9',
 27,
 'how to format 3*9 in calculator',
 '(3*9)',
 16,
 'square root of 16',
 'square root value of 16',
 '16 square root',
 'sqrt(1

In [26]:

labels


['Calculator',
 'Calculator',
 'TableSummarizer',
 'TableSummarizer',
 'TableSummarizer',
 'TableSummarizer',
 'TableSummarizer',
 'TableSummarizer',
 'Search',
 'Search',
 'Calculator',
 'Calculator',
 'Calculator',
 'Search',
 'Calculator',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'TableSummarizer',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Calculator',
 'Search',
 'Calculator',
 'Search',
 'Calculator',
 'Calculator',
 'Search',
 'Search',
 'Search',
 'Calculator']

In [27]:

texts2 = [ str(text) for text in texts]
texts  = texts2


In [28]:

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

dtm_pd = pd.DataFrame( X.toarray(), index=texts, columns=vectorizer.get_feature_names_out()  ).head(10)
dtm_pd


,10,100,16,27,albert,among,api,articles,biography,calculations,...,source,sqrt,square,step,summarization,the,to,tool,value,who
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
2+2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
{'csv_file': 'example.csv'},0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
{'csv_file': 'example.csv'},0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
{'csv_file': 'example.csv'},0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
{'csv_file': 'example.csv'},0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
{'csv_file': 'example.csv'},0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
{'csv_file': 'example.csv'},0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
{'query': 'online csv summarization tool'},0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.473289,0.0,0.0,0.525684,0.0,0.0
{'query': 'Google Colab pandas csv summarization'},0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.401900,0.0,0.0,0.000000,0.0,0.0


In [29]:



model = LogisticRegression()
model.fit(X, labels)

print("\n--- Model trained ---\n")



--- Model trained ---



In [30]:

# -------------------------------
# 6. TRAINED POLICY (PREDICT TOOL)
# -------------------------------
def predict_tool(query):
    X = vectorizer.transform([query])
    return model.predict(X)[0]


In [31]:

# -------------------------------
# 7. FINAL AGENT (USES TRAINED POLICY)
# -------------------------------
def run_agent(query):
    chosen_tool = predict_tool(query)

    print(f"\nQuery: {query}")
    print(f"Chosen Tool (trained): {chosen_tool}")

    if chosen_tool == "Calculator":
        result = calculator_tool(query)
    else:
        result = search_tool(query)

    print(f"Result: {result}")
    return result


In [32]:

# -------------------------------
# 8. TEST
# -------------------------------
print("\n--- Testing trained agent ---")

run_agent("5 * 6")
run_agent("Who invented electricity?")
run_agent("12 + 45")
run_agent("population of Japan")




--- Testing trained agent ---

Query: 5 * 6
Chosen Tool (trained): Search
Result: Searching for: 5 * 6

Query: Who invented electricity?
Chosen Tool (trained): Search
Result: Searching for: Who invented electricity?

Query: 12 + 45
Chosen Tool (trained): Search
Result: Searching for: 12 + 45

Query: population of Japan
Chosen Tool (trained): Search
Result: Searching for: population of Japan


'Searching for: population of Japan'



## PPO TRAINING LOOP



In [ ]:




print("\n--- TRAINING PPO ---\n")

for epoch in range(200):

    states = []
    actions = []
    rewards = []
    old_log_probs = []

    # collect batch
    for _ in range(16):
        q = random.choice(queries)
        s = encode(q)

        logits = policy(s)
        probs = torch.softmax(logits, dim=0)

        dist = torch.distributions.Categorical(probs)
        a = dist.sample()
        log_prob = dist.log_prob(a)

        # reward
        r = 1.0 if a.item() == correct_action(q) else -1.0

        states.append(s)
        actions.append(a)
        rewards.append(r)
        old_log_probs.append(log_prob.detach())


    states = torch.stack(states)
    actions = torch.stack(actions)
    old_log_probs = torch.stack(old_log_probs)


    # compute returns (simple, no baseline)
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns)

    # normalize
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)


    # PPO update
    logits = policy(states)
    probs = torch.softmax(logits, dim=1)
    dist = torch.distributions.Categorical(probs)

    new_log_probs = dist.log_prob(actions)

    ratio = torch.exp(new_log_probs - old_log_probs)

    surr1 = ratio * returns
    surr2 = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * returns

    loss = -torch.min(surr1, surr2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Training done.")







In [ ]:


def run_agent(query):
    s = encode(query)
    logits = policy(s)
    action = torch.argmax(logits).item()

    print(f"\nQuery: {query}")
    print(f"Chosen action: {['Calculator','Search'][action]}")

    if action == 0:
        result = calculator_tool(query)
    else:
        result = search_tool(query)

    print(f"Result: {result}")

